# Test the inverse problem

In this test we use a head model, produce an artificial signal and compute the estimated EEG using the forward matrix. We then implement the inverse solver and see if we reproduce the initial signal we created.

## Imports

In [44]:
%matplotlib qt

import mindmap as mm 
from pathlib import Path
import mne
import numpy as np

import pyxu.abc as pxa
import pyxu.operator as pxo 
import pyxu.opt.stop as pxst
from pyxu.opt.solver import PGD


config = mm.io.read_config("../configurations/spike.yml")

## Data

In [45]:
sub = "sub-pat10"

# files
mm_dir  = Path(config["MM_DIR"])
forward = mm_dir/sub/"fwd"/f"{sub}_surf-fwd.fif"
eeg     = mm_dir/sub/"eeg"/f"{sub}_ave.fif"
cst     = mm_dir/sub/"cst"/f"{sub}_cst-template.npy"

# data
forward = mne.read_forward_solution(forward)
eeg     = mne.read_evokeds(eeg)[0]
cst     = np.load(cst)

# plotting parameters
plot_kwargs = dict(
    subject = sub, 
    hemi = "both", 
    surface = "white",
    smoothing_steps = 10,
)

# pyxu parameters
pyxu_kwargs = dict(
    folder = "solverOutput/",
    exist_ok = True,
    writeback_rate = 50,
    verbosity = 1,
    show_progress = False,
    log_var = "x",
)

Reading forward solution from /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/fwd/sub-pat10_surf-fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523) not available
    Read EEG forward solution (6537 sources, 204 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Reading /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/eeg/sub-pat10_ave.fif ...
    Found the data of interest:
        t =       0.00 ...     996.00 ms (No comment)
        0 CTF compensation matrices available
        nave = 1 - aspect type = 100
No projector specified for this dataset. Please consider the method se

## Generate signal

In [46]:
# harmonic signal
hm = np.random.randint(0,10,3)

cs        = np.zeros(cst.shape[0])
cs[hm]    = 1
hm_signal = np.expand_dims(cst.T @ cs, axis=1)
hm_signal = hm_signal/np.abs(hm_signal.max())*10e-10      # scale to pico Amperes

# other signal
other = ...

# create stc
vertices = [forward["src"][0]["vertno"], forward["src"][1]["vertno"]]
activity = mne.SourceEstimate(hm_signal, vertices, tmin=0, tstep=0.1, subject=sub)

# visualize
activity.plot(**plot_kwargs)

Using control points [7.91595482e-10 8.46078549e-10 9.95321249e-10]


For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


## Apply forward

In [47]:
# forward
activity_eeg = mne.apply_forward(forward, activity, eeg.info)

# visualize
_ = activity_eeg.plot_topomap([0])

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]
Projecting source estimate to sensor space...
[done]


## Solve Inverse

In [48]:
# regularizing parameter
lam = 0.1

# fix orientation
fwd = forward.copy()
fwd = mne.convert_forward_solution(
    fwd,
    surf_ori=True,
    force_fixed=True,
    use_cps=True,
    verbose="warning",
)

# convert to arrays
fwd = fwd["sol"]["data"].astype("float64")
b   = activity_eeg.data.squeeze()

# normalize forward matrix and eeg recordings
#A_norms = np.linalg.norm(fwd, ord=2, axis=0)
#fwd     = fwd/A_norms[None,:]

#b_norm = np.linalg.norm(b, ord=2)
#b      = b/b_norm

# compute the adjoint (serves as initial guess)
fwd_inv = np.linalg.pinv(fwd)
x0      = fwd_inv@b

# write optimization problem
A = pxa.LinOp.from_array(fwd@cst.T)
F = pxo.SquaredL2Norm(dim_shape=A.codim_shape[0]).argshift(-b) * A     # fidelity
F.diff_lipschitz = F.estimate_diff_lipschitz()              
G = lam*pxo.L1Norm(dim_shape=x0.shape[0])                   # regularizer

opt = PGD(f=F, g=G, **pyxu_kwargs)

# define stopping criterion
stop_crit = pxst.RelError(eps=1e-5, var="x", f=None, norm=2) | pxst.MaxIter(5000)

# solve optimization problem
opt.fit(x0=x0, stop_crit=stop_crit, acceleration=True)
        
# convert solution to brain activity
alpha = opt.solution().squeeze()
x = cst.T@alpha
#x = mne.SourceEstimate(x*A_norms*b_norm, vertices, tmin=0, tstep=0.1, subject=sub)
#x = mne.SourceEstimate(x*b_norm, vertices, tmin=0, tstep=0.1, subject=sub)
#x = mne.SourceEstimate(x*A_norms, vertices, tmin=0, tstep=0.1, subject=sub)
x = mne.SourceEstimate(x, vertices, tmin=0, tstep=0.1, subject=sub)

In [49]:
x.plot(**plot_kwargs)

Using control points [7.16221472e-10 7.51977681e-10 8.89797328e-10]
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


In [38]:
import plotly.graph_objects as go

def plot_fourier_transform(x_hat):
    """
    Plot the graph fourier transform.

    Parameters
    ----------
        x_hat: nd.array (nb_nodes,)
            Graph fourier transform of x

    Returns
    -------
        fig: plotly.graph_objs
            figure for the graph fourier transform of x
    """
    trace = go.Bar(
        y=x_hat,
        marker=dict(
            color="darkblue",
            line=dict(color="darkblue"),
        ),
    )

    layout = go.Layout(dict(
        title="Fourier graph coefficients",
        showlegend=False, 
        plot_bgcolor="white",
        xaxis=dict(
            title="frequency indices",
            linecolor="black",
        ),
        yaxis=dict(
            title="amplitude",
            linecolor="black",
            gridcolor="black",
        ),
    ))

    return go.Figure(trace, layout=layout)

In [40]:
activity_cs = cst @ activity.data
plot_fourier_transform(activity_cs.squeeze())

In [39]:
x_cs = cst @ x.data
plot_fourier_transform(x_cs.squeeze())